# Profile-RAG — Half 2: Predict

Loads the FAISS vector DB built by `rag_profile_half1_embed.ipynb` and runs
Big Five trait prediction using a profile-aware RAG retriever.

**Requires:** `data/vector_db/essays_profile/` from Half 1.

Pipeline:
1. Profile the test set (label-blind) → `data/profile_db/essays_test/`
2. Install `ProfileRAGRetriever` — queries by profile similarity, returns trait-sliced exemplars.
3. Sanity check — render one few-shot context to confirm round-trip.
4. `predict` — run the classifier.
5. `evaluate` — score predictions.
6. Cleanup — restore original retriever.

In [ ]:
from pathlib import Path
import sys, os, json
from typing import Dict

import numpy as np
import pandas as pd

project_root = Path.cwd().resolve()
if not (project_root / "principle_model").exists():
    project_root = (project_root / ".." / "..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from rag.profiler.store import ProfileStore
from rag.profiler.runner import build_profiles
from rag.profiler.prompts import (
    FACETS,
    slice_profile_for_trait,
)
from rag.embedder import get_embedding
import rag.retriever as _retriever_mod
from rag.retriever import FeatureRAGRetriever

from principle_model.predict import predict
from principle_model.evaluate import evaluate

print("Project root:", project_root)

## Configuration

In [ ]:
# --- Paths ----------------------------------------------------------------
test_csv        = str(project_root / "data/split/essays/test50.csv")
principles_dir  = str(project_root / "data/principles")

test_profile_db = str(project_root / "data/profile_db/essays_test")     # generated below if missing
vector_db_dir   = str(project_root / "data/vector_db/essays_profile")   # built by Half 1

res_dir         = str(project_root / "result")
log_dir         = str(project_root / "log")

# --- Profiler model (for test set profiling) ------------------------------
profiler_model  = "gpt-4o-mini"

# --- Predictor model & prompt --------------------------------------------
model_name      = "gpt-4o-mini"
max_new_tokens  = 512
temperature     = 0.0
prompt_mode     = "rag_oneshot"   # rag_zeroshot | rag_oneshot | cot_rag_zeroshot | cot_rag_oneshot
top_k           = 3

TRAIT_NAMES = ("Openness to Experience", "Conscientiousness", "Extraversion", "Agreeableness", "Neuroticism")
TRAIT_CODES = {
    "Openness to Experience": "cOPN",
    "Conscientiousness":      "cCON",
    "Extraversion":           "cEXT",
    "Agreeableness":          "cAGR",
    "Neuroticism":            "cNEU",
}

test_df = pd.read_csv(test_csv)
print(f"Test  : {len(test_df):>4} rows  | mode={prompt_mode}  | top_k={top_k}")

# Confirm vector DB from Half 1 is present
for fname in ("vectors.faiss", "vectors_meta.jsonl"):
    p = Path(vector_db_dir) / fname
    if not p.exists():
        raise RuntimeError(f"Missing {p}. Run rag_profile_half1_embed.ipynb first.")
print(f"Vector DB found at: {vector_db_dir}")

## Step 1 — Profile the test set (label-blind)

Test profiles MUST be generated **without exposing labels** (`use_labels=False`)
to avoid leakage: at inference time the model would not know the ground truth.

In [ ]:
test_store_path = Path(test_profile_db) / "profile_store.jsonl"
test_store = ProfileStore(str(test_store_path))
test_store.load()
needed = len(test_df) - sum(
    1 for i in range(len(test_df)) if test_store.has(f"user_{i}") and test_store.get(f"user_{i}").get("valid")
)
print(f"Test profiles already in store: {len(test_store)}; missing: {needed}")

if needed > 0:
    test_store = build_profiles(
        data       = test_df,
        output_dir = test_profile_db,
        model_name = profiler_model,
        log_dir    = str(Path(log_dir) / "profiler_test"),
        use_labels = False,   # IMPORTANT: label-blind for test
    )
test_entries_by_idx = {
    int(e["user_id"].split("_")[1]): e for e in test_store.get_all() if e.get("valid")
}
print(f"Test profiles ready: {len(test_entries_by_idx)}")

## Step 2 — Profile-aware retriever adapter

Subclasses `FeatureRAGRetriever` to:
- embed the **test essay's profile** (not the raw essay) as the query,
- render retrieved exemplars as **trait-sliced profiles + label**.

Monkey-patched onto `rag.retriever.FeatureRAGRetriever` so that
`principle_model.predict` picks it up without any changes to predictor code.

In [ ]:
def render_full_profile_text(entry: Dict) -> str:
    """Deterministic rendering of a profile for embedding."""
    raw = entry.get("raw") or ""
    if raw.strip():
        return raw
    facets = entry.get("facets", {})
    ling   = entry.get("linguistic", {})
    lines = ["[FACETS]"]
    for code, name, *_ in FACETS:
        f = facets.get(code, {})
        lines.append(f"{code} {name:<18}| {f.get('signal','')} | {f.get('evidence','')}")
    lines.append("\n[LINGUISTIC]")
    for k, v in ling.items():
        lines.append(f"{k}: {v}")
    return "\n".join(lines)


class ProfileRAGRetriever(FeatureRAGRetriever):
    """Retriever that embeds the query essay's parsed profile (not raw text)
    and returns trait-sliced profile excerpts as few-shot exemplars.
    """

    def __init__(self, db_dir: str, test_profiles_by_idx: Dict[int, Dict], test_df: pd.DataFrame):
        super().__init__(db_dir=db_dir)
        self._use_finetuned = False
        self._test_profiles_by_idx = test_profiles_by_idx
        self._text_to_idx = {str(t): i for i, t in enumerate(test_df["text"].tolist())}

    def _embed_query_profile(self, query_text: str):
        idx = self._text_to_idx.get(str(query_text))
        if idx is None or idx not in self._test_profiles_by_idx:
            print(f"  [retriever] WARN: no test profile found for query (idx={idx}); falling back to raw text.")
            return self._embed(query_text)
        profile_text = render_full_profile_text(self._test_profiles_by_idx[idx])
        return np.array(self._embed(profile_text), dtype="float32")

    def build_similar_context(self, posts: str, trait: str, top_k: int = 3) -> str:
        trait_code = TRAIT_CODES.get(trait)
        if trait_code is None:
            return super().build_similar_context(posts=posts, trait=trait, top_k=top_k)

        query_emb = self._embed_query_profile(posts)
        all_results = self._search(query_emb, top_k * 4)

        blocks, seen = [], 0
        for r in all_results:
            if trait not in r.get("trait_labels", {}):
                continue
            label = r["trait_labels"][trait]
            features = r.get("features", {}) or {}
            profile = features.get("profile") or {}
            slice_text = slice_profile_for_trait(profile, trait_code) if profile else ""
            if not slice_text.strip():
                slice_text = "  (no profile slice available)"
            blocks.append(
                f"[Similar Profile {seen+1}] (label: {label})\n{slice_text}"
            )
            seen += 1
            if seen >= top_k:
                break
        return "\n\n".join(blocks)

In [ ]:
# Monkey-patch so principle_model.predict instantiates the profile-aware retriever
_OriginalRetriever = _retriever_mod.FeatureRAGRetriever

def _RetrieverFactory(db_dir=None):
    return ProfileRAGRetriever(
        db_dir=db_dir or vector_db_dir,
        test_profiles_by_idx=test_entries_by_idx,
        test_df=test_df,
    )

_retriever_mod.FeatureRAGRetriever = _RetrieverFactory
_retriever_mod.FINETUNED_ARTIFACTS_DIR = None
print("[adapter] ProfileRAGRetriever installed.")

## Step 3 — Sanity check: render one few-shot context

Confirms the retriever round-trips correctly for the first test essay.

In [ ]:
_smoke = _RetrieverFactory(db_dir=vector_db_dir)
_query_text = test_df.iloc[0]["text"]
for trait_full in TRAIT_NAMES:
    print(f"\n=== {trait_full} ===")
    print(_smoke.build_similar_context(posts=_query_text, trait=trait_full, top_k=top_k))

## Step 4 — Run prediction

In [ ]:
run_id, run_time, prediction_csv = predict(
    text_df        = test_df,
    model_name     = model_name,
    log_dir        = log_dir,
    prompt_mode    = prompt_mode,
    max_new_tokens = max_new_tokens,
    res_dir        = res_dir,
    temperature    = temperature,
    principles_dir = principles_dir,
    top_k          = top_k,
    vector_db_dir  = vector_db_dir,
)

print(f"\nDone in {run_time:.1f}s")
print(f"Predictions saved to: {prediction_csv}")

## Step 5 — Evaluate

In [ ]:
evaluation = evaluate(
    prediction_csv = prediction_csv,
    model_name     = model_name,
    res_dir        = res_dir,
    run_time       = run_time,
    prompt_mode    = prompt_mode,
    run_id         = run_id,
)

print("Summary CSV:", evaluation["summary_csv"])
print(f"Failed predictions: {evaluation['fail_count']} / {evaluation['n_records']}")
summary_df = pd.read_csv(evaluation["summary_csv"])
display(summary_df[["trait", "n_samples", "accuracy", "macro_f1", "weighted_f1"]]
        .sort_values("accuracy", ascending=False)
        .reset_index(drop=True))

## Step 6 — Restore the original retriever (cleanup)

Restores the original retriever class so subsequent cells / notebooks
aren't affected by the monkey-patch.

In [ ]:
_retriever_mod.FeatureRAGRetriever = _OriginalRetriever
print("[adapter] Original FeatureRAGRetriever restored.")